# Day 1 — Bookings Profile (Akhil)

**Goal:** Profile `data/raw/bookings.csv` — columns, dtypes, nulls, duplicates, date formats.

**Done-when:** These notes committed under `notebooks/`.

**Open questions to resolve:**
- Is `segment` a channel (Travel Agency/Direct/Corporate/Group/Walk-in) or customer-type field?
- Is occupancy tracked at room-night grain or booking grain?
- Does `total_rooms_available` actually exist in this file?
- Do all three files share a consistent date range?

In [ ]:
from pathlib import Path
import pandas as pd

# Resolve path relative to repo root regardless of CWD
REPO_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
BOOKINGS_PATH = REPO_ROOT / 'data' / 'raw' / 'bookings.csv'

print(f"Looking for: {BOOKINGS_PATH}")
print(f"Exists: {BOOKINGS_PATH.exists()}")

In [ ]:
df = pd.read_csv(BOOKINGS_PATH)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

## 1. Column names and dtypes

In [ ]:
df.dtypes.to_frame(name='dtype')

## 2. Null counts and percentages

In [ ]:
null_summary = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct':   (df.isnull().mean() * 100).round(2),
})
null_summary[null_summary['null_count'] > 0].sort_values('null_pct', ascending=False)

In [ ]:
# Total null counts across all columns
print(null_summary)

## 3. Duplicate rows

In [ ]:
n_exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {n_exact_dupes}")

# Duplicate reservation_ids (PK check)
n_dup_ids = df['reservation_id'].duplicated().sum() if 'reservation_id' in df.columns else 'N/A — column missing'
print(f"Duplicate reservation_id values: {n_dup_ids}")

## 4. Date column formats

In [ ]:
DATE_COLS = [c for c in ['booking_date', 'check_in_date', 'check_out_date'] if c in df.columns]

for col in DATE_COLS:
    sample_vals = df[col].dropna().unique()[:10]
    print(f"\n--- {col} ---")
    print(f"  dtype     : {df[col].dtype}")
    print(f"  samples   : {sample_vals}")
    
    # Try parsing to see if pandas can auto-detect format
    try:
        parsed = pd.to_datetime(df[col], infer_datetime_format=True, errors='coerce')
        n_failed = parsed.isnull().sum() - df[col].isnull().sum()
        print(f"  parse failures: {n_failed}")
        print(f"  range : {parsed.min()} → {parsed.max()}")
    except Exception as e:
        print(f"  parse error: {e}")

## 5. Segment column — value counts

In [ ]:
if 'segment' in df.columns:
    print("Segment value counts (raw):")
    print(df['segment'].value_counts(dropna=False))
    print(f"\nUnique segment values: {df['segment'].nunique()}")
else:
    print("WARNING: 'segment' column not found — update SPEC Section 4.")

## 6. Numeric columns — basic stats

In [ ]:
df.describe(include='all')

## 7. Schema contract check

Compare actual columns against SPEC Section 4 expected schema.

In [ ]:
EXPECTED_COLS = [
    'reservation_id', 'segment', 'room_type',
    'booking_date', 'check_in_date', 'check_out_date',
    'nights', 'rate', 'total_rooms_available',
]

actual_cols = set(df.columns.str.strip().str.lower())
expected_cols = set(c.lower() for c in EXPECTED_COLS)

missing  = expected_cols - actual_cols
extra    = actual_cols - expected_cols

print(f"Missing expected columns : {missing or 'None — all present'}")
print(f"Extra/unexpected columns : {extra or 'None'}")

# total_rooms_available is listed as open question in SPEC — flag it
if 'total_rooms_available' not in actual_cols:
    print("\nOPEN QUESTION: total_rooms_available not present — capacity must be assumed or derived.")

## 8. Profiling summary

Fill in after running all cells above.

In [ ]:
summary = {
    'total_rows':           df.shape[0],
    'total_columns':        df.shape[1],
    'actual_columns':       list(df.columns),
    'missing_from_spec':    list(missing) if 'missing' in dir() else 'run cell above',
    'exact_duplicate_rows': n_exact_dupes if 'n_exact_dupes' in dir() else 'run cell above',
    'duplicate_reservation_ids': n_dup_ids if 'n_dup_ids' in dir() else 'run cell above',
}

for k, v in summary.items():
    print(f"{k}: {v}")

---
## 9. Findings & open questions (fill in after running)

| Item | Finding |
|------|---------|
| Row count | |
| Columns present / missing | |
| Null columns | |
| Date format | |
| Segment type (channel vs customer) | |
| total_rooms_available present? | |
| Date range | |
| Duplicates | |